In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=65765a586c1c536627194f956fa526b0bdc597956f82232339ef3dd8fd44974a
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [3]:
# Generate Random number using |+> and measuring it
def generate_random_bit(n):
    qc = QuantumCircuit(n,n)

    #set to |+>
    for i in range(n):
      qc.h(i)
    #measure
    qc.measure(range(n),range(n))

    #run
    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    bitstring = list(counts.keys())[0]
    return [int(bit) for bit in bitstring]


In [4]:
def prepare_bits(bit,basis):
  qc = QuantumCircuit(1,1)
  if bit == 1:
    qc.x(0)

  if basis == 1:
    qc.h(0)
  return qc


In [5]:
def measure_qubit(qc,basis):
  if basis == 1:
    qc.h(0)

  #measure
  qc.measure(0,0)

  #run
  backend = BasicSimulator()
  compiled_circuit = transpile(qc, backend)
  job = backend.run(compiled_circuit, shots=1)
  result = job.result()
  counts = result.get_counts()

  bitstring = list(counts.keys())[0]
  return int(bitstring)

In [6]:
def create_shared_key(a_bits,a_bases,b_bases,b_results):
  a_key = []
  b_key = []
  matching_bit_pos = []

  #iterate against a_bits, if both a and b bases match, add to a and b keys and matching bit pos
  for i in range(len(a_bits)):
    if a_bases[i] == b_bases[i]:
      a_key.append(a_bits[i])
      b_key.append(b_results[i])
      matching_bit_pos.append(i)
  return a_key,b_key,matching_bit_pos

In [7]:
def show_results(a_bits, a_bases, b_bases, b_results, a_key, b_key, matching_positions):
    n = len(a_bits)

    #Transmission table
    print("\n" + "="*60)
    print("BB84 TRANSMISSION")
    print("="*60)

    rows = {
        "Position"  : [f"{i:2d}"             for i in range(n)],
        "A bit"     : [f" {b}"               for b in a_bits],
        "A basis"   : [f" {'X' if b else 'Z'}" for b in a_bases],
        "B basis"   : [f" {'X' if b else 'Z'}" for b in b_bases],
        "B result"  : [f" {r}"               for r in b_results],
        "Match"     : [" ✓" if i in matching_positions else " ✗" for i in range(n)],
    }

    for label, values in rows.items():
        print(f"{label:10}", " ".join(values))

    #Key comparison
    print("\n" + "="*60)
    print("FINAL KEY")
    print("="*60)
    print(f"\nA key: {a_key}")
    print(f"B key: {b_key}")

    errors = sum(a != b for a, b in zip(a_key, b_key))
    if errors == 0:
        print("\n✓ Keys match. Secure communication established.")
    else:
        print(f"\n✗ {errors} error(s) detected!")


In [8]:
def bb84_plain(n): #n = no of bits
  # generate bits using quantum measurement for a_bits, a_base  and b_base
  # a - generate random bits and bases using |+> measurement
  a_bits = generate_random_bit(n)
  a_bases = generate_random_bit(n)

  # b - choose random bases using \+> measurement
  b_bases   = generate_random_bit(n)
  b_results = [
        measure_qubit(prepare_bits(a_bits[i], a_bases[i]), b_bases[i])
        for i in range(n)
    ]

  # create shared key
  a_key, b_key, matching = create_shared_key(
        a_bits, a_bases, b_bases, b_results
    )

  # show result
  show_results(a_bits, a_bases, b_bases, b_results, a_key, b_key, matching)

  errors     = sum(a != b for a, b in zip(a_key, b_key))
  key_len    = len(a_key)
  error_rate = (errors / key_len * 100) if key_len else 0

  print("STATISTICS")
  print("="*60)
  print(f"Qubits sent:      {n}")
  print(f"Matching bases:   {len(matching)} ({len(matching)/n*100:.1f}%)")
  print(f"Key length:       {key_len} bits")
  print(f"QBER:             {error_rate:.2f}%")
  print("="*60)

  return a_key, b_key

In [9]:
alice, bob = bb84_plain(20)


BB84 TRANSMISSION
Position    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19
A bit       0  1  1  1  1  1  1  1  1  0  0  1  1  1  0  0  0  1  0  0
A basis     X  X  Z  Z  Z  X  Z  X  Z  X  Z  Z  Z  Z  Z  Z  Z  X  Z  Z
B basis     X  Z  X  X  X  Z  Z  Z  X  X  X  Z  Z  Z  Z  Z  Z  X  X  X
B result    0  1  1  0  1  0  1  0  0  0  1  1  1  1  0  0  0  1  0  1
Match       ✓  ✗  ✗  ✗  ✗  ✗  ✓  ✗  ✗  ✓  ✗  ✓  ✓  ✓  ✓  ✓  ✓  ✓  ✗  ✗

FINAL KEY

A key: [0, 1, 0, 1, 1, 1, 0, 0, 0, 1]
B key: [0, 1, 0, 1, 1, 1, 0, 0, 0, 1]

✓ Keys match. Secure communication established.
STATISTICS
Qubits sent:      20
Matching bases:   10 (50.0%)
Key length:       10 bits
QBER:             0.00%
